# 02 – Pobieranie danych pogodowych (Visual Crossing API)

Dla każdego meczu pobieram warunki pogodowe w miejscu i o godzinie jego rozpoczęcia z [Visual Crossing Timeline API](https://www.visualcrossing.com/weather-api).

- Klucz API jest czytany ze zmiennej środowiskowej `VISUAL_CROSSING_API_KEY`.
- Zbiór dzielę na 20 części. Każda jest zapisywana osobno, więc przerwane pobieranie można wznowić od ostatniej zapisanej części.
- Wyniki zawierają klucz meczu (`Division`, `Season`, `HomeTeam`, `AwayTeam`), dzięki czemu łączenie z meczami nie zależy od kolejności wierszy.

**Gotowy wynik jest już w repozytorium** (`data/external/visual_crossing_weather.csv`), więc ten notebook trzeba uruchamiać tylko przy ponownym pobieraniu danych.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

MATCHES_FILE = Path('../data/processed/matches_clean.csv')
CHUNKS_DIR = Path('../data/interim/weather_chunks')
WEATHER_FILE = Path('../data/external/visual_crossing_weather.csv')

API_KEY = os.getenv('VISUAL_CROSSING_API_KEY')
NUMBER_OF_CHUNKS = 20
KEY_COLS = ['Division', 'Season', 'HomeTeam', 'AwayTeam']

# Ustaw na True, aby pobrać dane od nowa mimo istniejącego pliku wynikowego
REFRESH = False

In [2]:
def get_weather_for_match(row, api_key):
    """Pobiera dane pogodowe dla pojedynczego meczu (wiersza DataFrame)."""
    if pd.isnull(row['Date']) or pd.isnull(row['Latitude']) or pd.isnull(row['Longitude']) or pd.isnull(row['Time']):
        return {}

    date_time = f"{row['Date']}T{row['Time']}"
    url = (
        'https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/'
        f"{row['Latitude']},{row['Longitude']}/{date_time}/{date_time}"
    )
    params = {'unitGroup': 'metric', 'include': 'current', 'key': api_key, 'contentType': 'json'}

    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f'Błąd sieci lub API dla wiersza {row.name}: {e}')
        return {}

    current_conditions = response.json().get('currentConditions') or {}
    return {
        'temp': current_conditions.get('temp'),
        'feelslike': current_conditions.get('feelslike'),
        'humidity': current_conditions.get('humidity'),
        'precip_mm': current_conditions.get('precip'),
        'precipprob': current_conditions.get('precipprob'),
        'preciptype': current_conditions.get('preciptype'),
        'windspeed_kph': current_conditions.get('windspeed'),
        'windgust_kph': current_conditions.get('windgust'),
        'pressure_hpa': current_conditions.get('pressure'),
        'cloudcover_percent': current_conditions.get('cloudcover'),
        'visibility_km': current_conditions.get('visibility'),
        'uvindex': current_conditions.get('uvindex'),
        'conditions': current_conditions.get('conditions'),
    }


def fetch_chunk(chunk_df, api_key, desc):
    results = []
    for _, row in tqdm(chunk_df.iterrows(), total=len(chunk_df), desc=desc):
        weather = get_weather_for_match(row, api_key)
        results.append({
            **row[KEY_COLS].to_dict(),
            'query_date': row['Date'],
            'query_time': row['Time'],
            'date_was_swapped': False,
            **weather,
        })
        time.sleep(0.1)  # nie przeciążam API
    return pd.DataFrame(results)

## Pobieranie w częściach

In [3]:
if WEATHER_FILE.exists() and not REFRESH:
    print(f'{WEATHER_FILE} już istnieje – pomijam pobieranie (ustaw REFRESH = True, aby pobrać ponownie).')
else:
    if not API_KEY:
        raise RuntimeError('Brak klucza API. Ustaw zmienną środowiskową VISUAL_CROSSING_API_KEY.')

    matches = pd.read_csv(MATCHES_FILE, low_memory=False)
    CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
    chunks = np.array_split(matches, NUMBER_OF_CHUNKS)

    for i, chunk_df in enumerate(chunks, start=1):
        chunk_file = CHUNKS_DIR / f'weather_chunk_{i}.csv'

        # Pomijam części pobrane wcześniej w całości – pozwala to wznowić przerwane pobieranie
        if chunk_file.exists() and len(pd.read_csv(chunk_file)) == len(chunk_df):
            print(f'Część {i}/{NUMBER_OF_CHUNKS}: już pobrana')
            continue

        fetch_chunk(chunk_df, API_KEY, desc=f'Część {i}/{NUMBER_OF_CHUNKS}').to_csv(chunk_file, index=False)

    # Łączę wszystkie części i sprawdzam, czy liczba wierszy zgadza się ze zbiorem meczów
    weather_df = pd.concat(
        [pd.read_csv(CHUNKS_DIR / f'weather_chunk_{i}.csv') for i in range(1, NUMBER_OF_CHUNKS + 1)],
        ignore_index=True,
    )
    assert len(weather_df) == len(matches), 'Liczba wierszy pogody nie zgadza się z liczbą meczów'

    weather_df.to_csv(WEATHER_FILE, index=False)
    print(f'Zapisano {len(weather_df)} wierszy do {WEATHER_FILE}')

../data/external/visual_crossing_weather.csv już istnieje – pomijam pobieranie (ustaw REFRESH = True, aby pobrać ponownie).
